# Transffered Trip Generation

In [1]:
import pickle
import pandas as pd
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import numpy as np

# ============================
# 1. Load Sydney Data
# ============================

df_sydney = pd.read_csv('Sydney_SA1_variables_for_trips.csv')

variables = [
    'Weighted Population',
    '%of commercial landuse',
    'station'
]

sydney_data = df_sydney[['SA1_CODE_2021'] + variables].copy() # SA1_CODE_2021 represents the spatial aggregation ID and should be replaced according to the spatial aggregation level used in your study.
sydney_data.dropna(inplace=True)

# ============================
# 2. Standardize Variables
# ============================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(sydney_data[variables])

# Convert back to DataFrame
X_scaled = pd.DataFrame(
    X_scaled,
    columns=[v + '_scaled' for v in variables],
    index=sydney_data.index
)

sydney_data = pd.concat([sydney_data, X_scaled], axis=1)

# ============================
# 3. Apply Seattle Coefficients
# ============================

beta_0 = 6.411
beta_wp = 0.182
beta_comm = 0.365
beta_station = 0.043

linear_part = (
    beta_0
    + beta_wp * sydney_data['Weighted Population_scaled']
    + beta_comm * sydney_data['%of commercial landuse_scaled']
    + beta_station * sydney_data['station_scaled']
)

# Negative Binomial with log link
sydney_data['Predicted_Weighted_Trips'] = np.exp(linear_part)

# ============================
# 4. Save Results
# ============================

predicted_sydney = sydney_data[['SA1_CODE_2021', 'Predicted_Weighted_Trips']]

# predicted_sydney.to_csv(
#     'Sydney_Predicted_Weighted_Trips_NegBin_StandardScaled.csv',
#     index=False
# )

print(predicted_sydney.head())
print("\n✅ Predictions completed successfully using StandardScaler!")

   SA1_CODE_2021  Predicted_Weighted_Trips
0    10102100707                529.713335
1    10102100710                602.135133
2    10104102401                496.200313
3    10104102402                476.143169
4    10104102412                479.572344

✅ Predictions completed successfully using StandardScaler!


In [2]:
predicted_sydney.to_csv(
    'SYD_Predicted_Weighted_Trips_NegBin_Seattle.csv',
    index=False
)